# Mapas Espaciais da Topologia, Infraestrutura e Alocação de VEs (Alimentador ESB01S4)
## Estudo de Caso: Rede Real de Distribuição MT/BT em Eusébio/CE (Co-simulação Mosaik / OpenDSS / Caldera ICM)

Este notebook é dedicado exclusivamente à geração e exportação dos mapas georreferenciados da topologia da rede elétrica em altíssima resolução (600 DPI e PDF vetorial) na pasta `output/figuras/figuras_topologia/`, separados individualmente para facilitar a referenciação no LaTeX:
1. **Figura 1a (`fig1a_topologia_barras_alimentador`):** Traçado físico das redes de Média Tensão (13,8 kV) e Baixa Tensão (220/380 V) com suas respectivas barras;
2. **Figura 1b (`fig1b_alocacao_equipamentos_alimentador`):** Dispersão espacial dos 40 transformadores MT/BT, 303 sistemas fotovoltaicos (GD FV), 947 residências BT e Subestação SE Eusébio;
3. **Figura 1a - Satélite (`fig1a_topologia_barras_alimentador_satelite`):** Topologia e barras por nível de tensão sobrepostas à fotografia de satélite real (*Esri World Imagery*, zoom 16, $\alpha = 0,65$);
4. **Figura 1b - Satélite (`fig1b_alocacao_equipamentos_alimentador_satelite`):** Alocação espacial de equipamentos e cargas sobreposta à fotografia de satélite real (*Esri World Imagery*, zoom 16, $\alpha = 0,65$);
5. **Figura 2 (`fig2_alocacao_veiculos_eletricos_50pct`):** Mapa de alocação espacial estocástica da frota de 474 veículos elétricos (2.581,2 kW) no cenário de 50% de penetração residencial BT (Seed 2026).

**Autoria:** Paulo Victor Pereira Lima  
*Grupo de Redes Elétricas Inteligentes (GREI) / Departamento de Engenharia Elétrica / Universidade Federal do Ceará (UFC)*

In [13]:
import os
import sys
import json
import math
import io
import re
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.path as mpath

# Localização robusta da raiz do projeto
curr = Path.cwd().resolve()
while curr.parent != curr:
    if (curr / "pyproject.toml").exists():
        PROJECT_ROOT = curr
        break
    curr = curr.parent
else:
    PROJECT_ROOT = Path("..").resolve()

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

OUTPUT_DIR = PROJECT_ROOT / "output"
TABELAS_DIR = OUTPUT_DIR / "tabelas"
TOPOLOGIA_DIR = OUTPUT_DIR / "topologia"
FIGURAS_BASE_DIR = OUTPUT_DIR / "figuras"
FIGURAS_TOPO_DIR = FIGURAS_BASE_DIR / "figuras_topologia"
FIGURAS_TOPO_DIR.mkdir(parents=True, exist_ok=True)

# Estilo visual acadêmico para LaTeX / ABNT / IEEE (fontes ampliadas e moldura nítida)
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 13,
    "axes.labelsize": 15,
    "axes.titlesize": 15.5,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 12,
    "figure.titlesize": 16,
    "axes.edgecolor": "black",
    "axes.linewidth": 1.4,
})

def estilizar_eixos_topologia(ax, sob_satelite=False):
    """Aplica moldura preta nos 4 lados, ticks a cada 0.5 km (sub-ticks a cada 0.25 km) e padroniza fontes."""
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor("black")
        spine.set_linewidth(1.4)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(0.5))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.5))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.25))
    ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.25))
    ax.tick_params(
        axis="both", which="major", direction="out",
        length=6.5, width=1.3, colors="black", labelsize=13,
        bottom=True, left=True, top=True, right=True,
    )
    ax.tick_params(
        axis="both", which="minor", direction="out",
        length=3.5, width=1.0, colors="black",
        bottom=True, left=True, top=True, right=True,
    )
    if sob_satelite:
        ax.grid(True, which="major", linestyle=":", color="white", alpha=0.45, linewidth=0.9)
        ax.grid(True, which="minor", linestyle=":", color="white", alpha=0.22, linewidth=0.6)
    else:
        ax.grid(True, which="major", linestyle=":", color="#777777", alpha=0.55, linewidth=0.9)
        ax.grid(True, which="minor", linestyle=":", color="#AAAAAA", alpha=0.30, linewidth=0.6)

def salvar_figura_topologia(fig, nome_arquivo, dpi=600):
    caminho_png = FIGURAS_TOPO_DIR / f"{nome_arquivo}.png"
    caminho_pdf = FIGURAS_TOPO_DIR / f"{nome_arquivo}.pdf"
    fig.savefig(caminho_png, dpi=dpi, bbox_inches="tight")
    fig.savefig(caminho_pdf, bbox_inches="tight")
    print(f"Salvo em Topologia: {caminho_png.name} e {caminho_pdf.name} ({dpi} DPI)")

# Carregamento dos metadados do transformador e barra mais críticos
df_ranking_trafos = pd.read_csv(TABELAS_DIR / "tabela_transformadores_ranking_criticos.csv")
trafo_mais_critico = df_ranking_trafos.iloc[0]["trafo"]
s_nom_critico = float(df_ranking_trafos.iloc[0]["s_nom_kva"])

with open(TOPOLOGIA_DIR / "barra_bt_mais_afetada.json", "r", encoding="utf-8") as f:
    info_barra = json.load(f)

print("Ambiente de Topologia configurado com sucesso!")
print(f"Raiz do Projeto      : {PROJECT_ROOT}")
print(f"Pasta de Topologia   : {TOPOLOGIA_DIR}")
print(f"Saída das Figuras    : {FIGURAS_TOPO_DIR}")
print(f"Barra BT Crítica     : {info_barra['bus']} ({info_barra['distancia_km']:.2f} km da SE)")
print(f"Trafo Mais Crítico   : {trafo_mais_critico} ({s_nom_critico:.0f} kVA)")

Ambiente de Topologia configurado com sucesso!
Raiz do Projeto      : /mnt/c/Users/pvict/OneDrive/TCC/ev-analysis-2026
Pasta de Topologia   : /mnt/c/Users/pvict/OneDrive/TCC/ev-analysis-2026/output/topologia
Saída das Figuras    : /mnt/c/Users/pvict/OneDrive/TCC/ev-analysis-2026/output/figuras/figuras_topologia
Barra BT Crítica     : 3864601538458035 (7.33 km da SE)
Trafo Mais Crítico   : TRF_29L2218A (75 kVA)


## Módulo 1: Topologia e Infraestrutura Física da Rede ESB01S4 (Figuras 1a e 1b)

Mapas espaciais georreferenciados gerados em figuras separadas para inclusão direta no LaTeX:
- **Figura 1a (`fig1a_topologia_barras_alimentador`):** Traçado físico das linhas e localização de todas as barras da rede classificadas por nível de tensão: Média Tensão (MT - 13,8 kV) em **vermelho** e Baixa Tensão (BT - 220/380 V) em **azul-escuro**;
- **Figura 1b (`fig1b_alocacao_equipamentos_alimentador`):** Dispersão espacial dos 40 transformadores MT/BT (**triângulos em roxo vibrante #7719FF**), 303 sistemas fotovoltaicos (**quadrados verdes**), 947 residências BT (**círculos laranja-avermelhados**) e Subestação SE Eusébio (**estrela vermelha**).

In [14]:
# FIGURAS 1A e 1B: Topologia Georreferenciada e Infraestrutura Física da Rede ESB01S4 (Figuras Individuais)

# 1. Carrega o grafo topológico e propaga coordenadas geodésicas reais
caminho_topo = TOPOLOGIA_DIR / "topologia_ESB01S4_TCC.json"
if not caminho_topo.exists():
    caminho_topo = PROJECT_ROOT / "data" / "rede" / "ESB01S4" / "topologia_ESB01S4_TCC.json"

with open(caminho_topo, "r", encoding="utf-8") as f:
    topo = json.load(f)

nodes = topo["nodes"]
edges = topo["edges"]
nodes_map = {n["id"].lower(): n for n in nodes}

# Coordenada base da Subestação (pac_esb01s4)
sub_node = next(n for n in nodes if n["node_type"] == "refbus")
lon0 = sub_node["metadata"]["x"]
lat0 = sub_node["metadata"]["y"]

# Propagação de coordenadas para 100% dos nós
coords_map = {
    n["id"].lower(): (n["metadata"]["x"], n["metadata"]["y"])
    for n in nodes
    if n["metadata"].get("coord_defined", False)
}
changed = True
while changed:
    changed = False
    for e in edges:
        s, t = e["source"].lower(), e["target"].lower()
        if s in coords_map and t not in coords_map:
            coords_map[t] = coords_map[s]
            changed = True
        elif t in coords_map and s not in coords_map:
            coords_map[s] = coords_map[t]
            changed = True

# Conversão para quilômetros relativos à Subestação (Eusébio/CE)
coords_km = {}
for b, (x, y) in coords_map.items():
    dx_km = (x - lon0) * 111.065
    dy_km = (y - lat0) * 110.574
    coords_km[b] = (dx_km, dy_km)

# 2. Separação de segmentos de linha: Média Tensão (MT) vs. Baixa Tensão (BT)
mt_segments = []
bt_segments = []

for e in edges:
    s, t = e["source"].lower(), e["target"].lower()
    if s not in coords_km or t not in coords_km:
        continue
    xs = (coords_km[s][0], coords_km[t][0])
    ys = (coords_km[s][1], coords_km[t][1])
    if xs[0] == xs[1] and ys[0] == ys[1]:
        continue

    is_mt = (
        nodes_map[s]["metadata"].get("kv_base", 0) > 1.0
        or nodes_map[t]["metadata"].get("kv_base", 0) > 1.0
        or "smt_" in e.get("id", "").lower()
    )
    if is_mt and e.get("edge_type") != "transformer":
        mt_segments.append((xs, ys))
    else:
        bt_segments.append((xs, ys))

# 3. Leitura dos Geradores Solares Fotovoltaicos (303 PVs em ESB01S4_pv.dss)
caminho_pv_dss = PROJECT_ROOT / "data" / "rede" / "ESB01S4" / "ESB01S4_pv.dss"
pvs_data = []
if caminho_pv_dss.exists():
    with open(caminho_pv_dss, "r", encoding="utf-8") as f:
        for l in f:
            if "new pvsystem" in l.lower():
                m_bus = re.search(r"bus1=([^\s]+)", l, re.I)
                m_kva = re.search(r"kva=([^\s]+)", l, re.I)
                if m_bus:
                    b_pv = m_bus.group(1).split(".")[0].lower()
                    kva = float(m_kva.group(1)) if m_kva else 5.0
                    if b_pv in coords_km:
                        pvs_data.append({"bus": b_pv, "kva": kva, "x": coords_km[b_pv][0], "y": coords_km[b_pv][1]})
df_pvs_plot = pd.DataFrame(pvs_data)

# 4. Leitura dos 40 Transformadores MT/BT
caminhos_trf = list((PROJECT_ROOT / "data" / "rede" / "ESB01S4").glob("TransformadorMTMTMTBT*.dss"))
trafos_data = []
if caminhos_trf:
    with open(caminhos_trf[0], "r", encoding="utf-8", errors="ignore") as f:
        for l in f:
            m = re.search(r'New "Transformer\.([^"]+)".*?buses=\[([^\]]+)\].*?kvas=\[([^\]]+)\]', l, re.I)
            if m:
                trf_nome = m.group(1).strip()
                barras = [b.strip(' "\'\t').split(".")[0].lower() for b in m.group(2).split()]
                kvas = [float(k) for k in m.group(3).split()]
                b_sec = barras[1] if len(barras) > 1 else barras[0]
                coord = coords_km.get(b_sec, coords_km.get(barras[0]))
                if coord:
                    trafos_data.append({"nome": trf_nome, "bus": b_sec, "kva": kvas[0], "x": coord[0], "y": coord[1]})
df_trafos_plot = pd.DataFrame(trafos_data)

# Identificação da Barra Crítica BT e Trafo Crítico do TCC
b_crit_id = str(info_barra["bus"]).lower()
b_crit_coord = coords_km.get(b_crit_id)
trf_crit_coord = None
if not df_trafos_plot.empty:
    sub_crit_trf = df_trafos_plot[df_trafos_plot["nome"].str.upper() == trafo_mais_critico.upper()]
    if not sub_crit_trf.empty:
        trf_crit_coord = (sub_crit_trf.iloc[0]["x"], sub_crit_trf.iloc[0]["y"])

# 5. Leitura das Cargas Residenciais BT do Alimentador (947 Domicílios)
cargas_data = []
p_carga_bt = PROJECT_ROOT / "data" / "rede" / "ESB01S4" / "CargasBT_Sintetico_20250139_ESB01S4.dss"
if not p_carga_bt.exists():
    p_cands = list((PROJECT_ROOT / "data" / "rede" / "ESB01S4").glob("*CargasBT*.dss"))
    p_carga_bt = p_cands[0] if p_cands else None

cargas_dict = {}
if p_carga_bt and p_carga_bt.exists():
    with open(p_carga_bt, "r", encoding="utf-8", errors="ignore") as f:
        for l in f:
            if "new" in l.lower() and "load." in l.lower() and "load.ev_" not in l.lower():
                m_name = re.search(r"load\.([^\\s\"]+)", l, re.I)
                m_bus = re.search(r"bus1=[\"']?([^\\s\"']+)", l, re.I)
                m_kw = re.search(r"kw\\s*=\\s*([0-9\\.]+)", l, re.I)
                if m_name and m_bus:
                    nome = m_name.group(1)
                    uc_id = nome.rsplit("_", 1)[0]
                    bus = m_bus.group(1).replace('"', '').replace("'", '').split(".")[0].lower()
                    kw = float(m_kw.group(1)) if m_kw else 1.0
                    if uc_id not in cargas_dict:
                        cargas_dict[uc_id] = {"bus": bus, "kw": kw}
                    else:
                        cargas_dict[uc_id]["kw"] += kw

for uc_id, c in cargas_dict.items():
    b = c["bus"]
    if b in coords_km:
        cargas_data.append({"uc_id": uc_id, "bus": b, "kw": c["kw"], "x": coords_km[b][0], "y": coords_km[b][1]})
df_cargas_plot = pd.DataFrame(cargas_data)

# Classificação das barras por nível de tensão (MT vs. BT)
mt_buses_pts = []
bt_buses_pts = []
for n in nodes:
    b_id = n["id"].lower()
    if b_id in coords_km:
        is_node_mt = n["metadata"].get("kv_base", 0) > 1.0 or n["node_type"] == "refbus"
        if is_node_mt:
            mt_buses_pts.append(coords_km[b_id])
        else:
            bt_buses_pts.append(coords_km[b_id])

# =========================================================================
# FIGURA 1A: TOPOLOGIA DA REDE E BARRAS POR NÍVEL DE TENSÃO
# =========================================================================
fig_1a, ax_1 = plt.subplots(figsize=(11.5, 14.2), dpi=600)

for idx, (xs, ys) in enumerate(bt_segments):
    lbl = "Rede BT (220/380 V)" if idx == 0 else ""
    ax_1.plot(xs, ys, color="#0C4B8E", linewidth=2.0, alpha=1.0, label=lbl, zorder=1)

for idx, (xs, ys) in enumerate(mt_segments):
    lbl = "Rede MT (13,8 kV)" if idx == 0 else ""
    ax_1.plot(xs, ys, color="#D73027", linewidth=2.0, alpha=1.0, label=lbl, zorder=2)

if bt_buses_pts:
    bt_xs = [pt[0] for pt in bt_buses_pts]
    bt_ys = [pt[1] for pt in bt_buses_pts]
    ax_1.scatter(bt_xs, bt_ys, s=22, color="#0C4B8E", edgecolor="none", linewidth=0, alpha=1.0, label=f"Barras BT ({len(bt_xs):,} barras)", zorder=3)

if mt_buses_pts:
    mt_xs = [pt[0] for pt in mt_buses_pts]
    mt_ys = [pt[1] for pt in mt_buses_pts]
    ax_1.scatter(mt_xs, mt_ys, s=22, color="#D73027", edgecolor="none", linewidth=0, alpha=1.0, label=f"Barras MT ({len(mt_xs):,} barras)", zorder=4)

ax_1.scatter([0], [0], s=450, marker="*", color="#990000", edgecolor="black", linewidth=1.4, zorder=8, label="Subestação SE (13,8 kV)")

ax_1.set_title("Topologia da Rede e Barras por Nível de Tensão", fontweight="bold", pad=12, fontsize=15.5)
ax_1.set_xlabel("Distância Leste–Oeste relativa à Subestação (km)", fontweight="bold", fontsize=15)
ax_1.set_ylabel("Distância Norte–Sul relativa à Subestação (km)", fontweight="bold", fontsize=15)
ax_1.legend(loc="lower left", frameon=True, fontsize=12, framealpha=0.95, edgecolor="black", labelspacing=1.0, handletextpad=0.8, markerscale=0.85, borderpad=1.0)
ax_1.set_aspect("equal", adjustable="datalim")
estilizar_eixos_topologia(ax_1, sob_satelite=False)

plt.tight_layout()
fig_1a.canvas.draw()
xlim_ref_padrao = ax_1.get_xlim()
ylim_ref_padrao = ax_1.get_ylim()

salvar_figura_topologia(fig_1a, "fig1a_topologia_barras_alimentador")
plt.show()

# =========================================================================
# FIGURA 1B: INFRAESTRUTURA FÍSICA, EQUIPAMENTOS E CARGAS
# =========================================================================
fig_1b, ax_2 = plt.subplots(figsize=(11.5, 14.2), dpi=600)

for idx, (xs, ys) in enumerate(bt_segments):
    lbl = "Rede BT (220/380 V)" if idx == 0 else ""
    ax_2.plot(xs, ys, color="#0C4B8E", linewidth=2.0, alpha=1.0, label=lbl, zorder=1)

for idx, (xs, ys) in enumerate(mt_segments):
    lbl = "Rede MT (13,8 kV)" if idx == 0 else ""
    ax_2.plot(xs, ys, color="#D73027", linewidth=2.0, alpha=1.0, label=lbl, zorder=2)

if not df_cargas_plot.empty:
    ax_2.scatter(
        df_cargas_plot["x"],
        df_cargas_plot["y"],
        s=42,
        marker="o",
        facecolor="#FF4500",
        edgecolor="#801500",
        linewidth=0.7,
        alpha=1.0,
        label=f"Residências BT ({len(df_cargas_plot)} domicílios)",
        zorder=4,
    )

if not df_pvs_plot.empty:
    ax_2.scatter(
        df_pvs_plot["x"],
        df_pvs_plot["y"],
        s=50,
        marker="s",
        color="#00C853",
        edgecolor="#004D1A",
        linewidth=0.7,
        alpha=1.0,
        zorder=5,
        label=f"Geração Solar GD FV ({len(df_pvs_plot)} PVs)",
    )

if not df_trafos_plot.empty:
    ax_2.scatter(
        df_trafos_plot["x"],
        df_trafos_plot["y"],
        s=115,
        marker="^",
        color="#7719FF",
        edgecolor="#3B0780",
        linewidth=0.9,
        alpha=1.0,
        zorder=6,
        label=f"Transformadores MT/BT ({len(df_trafos_plot)} trafos)",
    )

ax_2.scatter([0], [0], s=450, marker="*", color="#990000", edgecolor="black", linewidth=1.4, zorder=8, label="Subestação SE (13,8 kV)")

ax_2.set_title("Alocação Espacial de Equipamentos e Cargas", fontweight="bold", pad=12, fontsize=15.5)
ax_2.set_xlabel("Distância Leste–Oeste relativa à Subestação (km)", fontweight="bold", fontsize=15)
ax_2.set_ylabel("Distância Norte–Sul relativa à Subestação (km)", fontweight="bold", fontsize=15)
ax_2.legend(loc="lower left", frameon=True, fontsize=12, framealpha=0.95, edgecolor="black", labelspacing=1.0, handletextpad=0.8, markerscale=0.85, borderpad=1.0)
ax_2.set_aspect("equal", adjustable="box")
ax_2.set_xlim(xlim_ref_padrao)
ax_2.set_ylim(ylim_ref_padrao)
estilizar_eixos_topologia(ax_2, sob_satelite=False)

plt.tight_layout()
salvar_figura_topologia(fig_1b, "fig1b_alocacao_equipamentos_alimentador")
plt.show()
print("Figuras 1a e 1b (limpas) salvas com sucesso em 600 DPI (PNG e PDF)!")

Salvo em Topologia: fig1a_topologia_barras_alimentador.png e fig1a_topologia_barras_alimentador.pdf (600 DPI)


Salvo em Topologia: fig1b_alocacao_equipamentos_alimentador.png e fig1b_alocacao_equipamentos_alimentador.pdf (600 DPI)


Figuras 1a e 1b (limpas) salvas com sucesso em 600 DPI (PNG e PDF)!


## Módulo 2: Topologia e Infraestrutura com Imagem de Satélite Real (Figuras 1a e 1b - Satélite)

Cópias exatas das Figuras 1a e 1b sobrepostas individualmente à fotografia aérea de satélite em alta resolução (*Esri World Imagery*, zoom 16, com transparência $\alpha = 0,65$) do município de Eusébio/CE.

In [17]:
# FIGURAS 1A e 1B (Satélite): Figuras Individuais com Mapa Real Esri World Imagery

# =========================================================================
# PARÂMETROS DE CONFIGURAÇÃO DO SATÉLITE E DO ZOOM
# =========================================================================
ZOOM_SATELITE = 16       # Nitidez Esri: 15 (média), 16 (alta), 17 (ultra-alta)
ALPHA_SATELITE = 0.65    # Transparência da foto (0.0 = invisível, 1.0 = sem transparência)
FATOR_ZOOM = 1.0         # 1.0 = enquadramento idêntico às Figuras 1a/1b limpas; > 1.0 aproxima (ex: 1.25); < 1.0 afasta (ex: 0.90)
DESLOCAR_X_KM = 0.0      # Deslocamento horizontal opcional do centro de visão (km)
DESLOCAR_Y_KM = 0.0      # Deslocamento vertical opcional do centro de visão (km)

# 1. Funções matemáticas para projeção Slippy Map / Web Mercator
def deg2num(lat_deg, lon_deg, zoom):
    lat_rad = math.radians(lat_deg)
    n = 2.0 ** zoom
    xtile = int((lon_deg + 180.0) / 360.0 * n)
    ytile = int((1.0 - math.asinh(math.tan(lat_rad)) / math.pi) / 2.0 * n)
    return xtile, ytile

def num2deg(xtile, ytile, zoom):
    n = 2.0 ** zoom
    lon_deg = xtile / n * 360.0 - 180.0
    lat_rad = math.atan(math.sinh(math.pi * (1.0 - 2.0 * ytile / n)))
    lat_deg = math.degrees(lat_rad)
    return lat_deg, lon_deg

def baixar_tile_esri(args):
    x, y, zoom = args
    url = f"https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{zoom}/{y}/{x}"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"})
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            return (x, y, resp.read())
    except Exception:
        return (x, y, None)

def carregar_mapa_satelite_alimentador(lon0, lat0, coords_km, cache_dir, zoom=16, pad_km=0.40):
    caminho_cache_img = cache_dir / f"satelite_esb01s4_z{zoom}.png"
    caminho_cache_meta = cache_dir / f"satelite_esb01s4_z{zoom}_meta.json"
    xs = [pt[0] for pt in coords_km.values()]
    ys = [pt[1] for pt in coords_km.values()]
    x_min_km, x_max_km = min(xs), max(xs)
    y_min_km, y_max_km = min(ys), max(ys)
    lon_min = lon0 + (x_min_km - pad_km) / 111.065
    lon_max = lon0 + (x_max_km + pad_km) / 111.065
    lat_min = lat0 + (y_min_km - pad_km) / 110.574
    lat_max = lat0 + (y_max_km + pad_km) / 110.574
    xtile_w, ytile_n = deg2num(lat_max, lon_min, zoom)
    xtile_e, ytile_s = deg2num(lat_min, lon_max, zoom)
    lat_top, lon_left = num2deg(xtile_w, ytile_n, zoom)
    lat_bottom, lon_right = num2deg(xtile_e + 1, ytile_s + 1, zoom)
    extent_km = [
        (lon_left - lon0) * 111.065,
        (lon_right - lon0) * 111.065,
        (lat_bottom - lat0) * 110.574,
        (lat_top - lat0) * 110.574
    ]
    if caminho_cache_img.exists() and caminho_cache_meta.exists():
        try:
            with open(caminho_cache_meta, "r", encoding="utf-8") as f:
                meta = json.load(f)
            img = Image.open(caminho_cache_img)
            return img, meta["extent_km"]
        except Exception:
            pass
    tarefas = [(x, y, zoom) for y in range(ytile_n, ytile_s + 1) for x in range(xtile_w, xtile_e + 1)]
    tiles_dict = {}
    with ThreadPoolExecutor(max_workers=8) as executor:
        for x, y, data in executor.map(baixar_tile_esri, tarefas):
            if data:
                tiles_dict[(x, y)] = data
    num_cols = xtile_e - xtile_w + 1
    num_rows = ytile_s - ytile_n + 1
    img_montada = Image.new("RGB", (num_cols * 256, num_rows * 256))
    for y_idx, y in enumerate(range(ytile_n, ytile_s + 1)):
        for x_idx, x in enumerate(range(xtile_w, xtile_e + 1)):
            if (x, y) in tiles_dict:
                tile_img = Image.open(io.BytesIO(tiles_dict[(x, y)]))
                img_montada.paste(tile_img, (x_idx * 256, y_idx * 256))
    try:
        caminho_cache_img.parent.mkdir(parents=True, exist_ok=True)
        img_montada.save(caminho_cache_img)
        with open(caminho_cache_meta, "w", encoding="utf-8") as f:
            json.dump({"extent_km": extent_km, "zoom": zoom}, f, indent=2)
    except Exception:
        pass
    return img_montada, extent_km

# 2. Obtenção da imagem do satélite (ou carregamento direto do cache local)
cache_dir = PROJECT_ROOT / "data" / "rede" / "ESB01S4"
img_sat, extent_sat = carregar_mapa_satelite_alimentador(lon0, lat0, coords_km, cache_dir, zoom=ZOOM_SATELITE)

# Cálculo dos limites de enquadramento conforme FATOR_ZOOM
x0_ref, x1_ref = xlim_ref_padrao
y0_ref, y1_ref = ylim_ref_padrao
cx_sat = 0.5 * (x0_ref + x1_ref) + DESLOCAR_X_KM
cy_sat = 0.5 * (y0_ref + y1_ref) + DESLOCAR_Y_KM
hw_sat = 0.5 * (x1_ref - x0_ref) / FATOR_ZOOM
hh_sat = 0.5 * (y1_ref - y0_ref) / FATOR_ZOOM
xlim_sat = (cx_sat - hw_sat, cx_sat + hw_sat)
ylim_sat = (cy_sat - hh_sat, cy_sat + hh_sat)

# =========================================================================
# FIGURA 1A (SATÉLITE): TOPOLOGIA DA REDE E BARRAS POR NÍVEL DE TENSÃO
# =========================================================================
fig_1a_sat, ax_1s = plt.subplots(figsize=(11.5, 14.2), dpi=600)

for idx, (xs, ys) in enumerate(bt_segments):
    lbl = "Rede BT (220/380 V)" if idx == 0 else ""
    ax_1s.plot(xs, ys, color="#0C4B8E", linewidth=2.0, alpha=1.0, label=lbl, zorder=2)

for idx, (xs, ys) in enumerate(mt_segments):
    lbl = "Rede MT (13,8 kV)" if idx == 0 else ""
    ax_1s.plot(xs, ys, color="#D73027", linewidth=2.0, alpha=1.0, label=lbl, zorder=3)

if bt_buses_pts:
    bt_xs = [pt[0] for pt in bt_buses_pts]
    bt_ys = [pt[1] for pt in bt_buses_pts]
    ax_1s.scatter(bt_xs, bt_ys, s=22, color="#0C4B8E", edgecolor="none", linewidth=0, alpha=1.0, label=f"Barras BT ({len(bt_xs):,} barras)", zorder=4)

if mt_buses_pts:
    mt_xs = [pt[0] for pt in mt_buses_pts]
    mt_ys = [pt[1] for pt in mt_buses_pts]
    ax_1s.scatter(mt_xs, mt_ys, s=22, color="#D73027", edgecolor="none", linewidth=0, alpha=1.0, label=f"Barras MT ({len(mt_xs):,} barras)", zorder=5)

ax_1s.scatter([0], [0], s=450, marker="*", color="#990000", edgecolor="black", linewidth=1.4, zorder=8, label="Subestação SE (13,8 kV)")

ax_1s.autoscale(False)
ax_1s.imshow(img_sat, extent=extent_sat, origin="upper", alpha=ALPHA_SATELITE, zorder=0)
ax_1s.set_aspect("equal", adjustable="box")
ax_1s.set_xlim(xlim_sat)
ax_1s.set_ylim(ylim_sat)

ax_1s.set_title("Topologia da Rede e Barras por Nível de Tensão", fontweight="bold", pad=12, fontsize=15.5)
ax_1s.set_xlabel("Distância Leste–Oeste relativa à Subestação (km)", fontweight="bold", fontsize=15)
ax_1s.set_ylabel("Distância Norte–Sul relativa à Subestação (km)", fontweight="bold", fontsize=15)
ax_1s.legend(loc="lower left", frameon=True, fontsize=12, framealpha=0.95, edgecolor="black", labelspacing=1.0, handletextpad=0.8, markerscale=0.85, borderpad=1.0)
estilizar_eixos_topologia(ax_1s, sob_satelite=True)

plt.tight_layout()
salvar_figura_topologia(fig_1a_sat, "fig1a_topologia_barras_alimentador_satelite")
plt.show()

# =========================================================================
# FIGURA 1B (SATÉLITE): INFRAESTRUTURA FÍSICA E EQUIPAMENTOS
# =========================================================================
fig_1b_sat, ax_2s = plt.subplots(figsize=(11.5, 14.2), dpi=600)

for idx, (xs, ys) in enumerate(bt_segments):
    lbl = "Rede BT (220/380 V)" if idx == 0 else ""
    ax_2s.plot(xs, ys, color="#0C4B8E", linewidth=2.0, alpha=1.0, label=lbl, zorder=2)

for idx, (xs, ys) in enumerate(mt_segments):
    lbl = "Rede MT (13,8 kV)" if idx == 0 else ""
    ax_2s.plot(xs, ys, color="#D73027", linewidth=2.0, alpha=1.0, label=lbl, zorder=3)

if not df_cargas_plot.empty:
    ax_2s.scatter(
        df_cargas_plot["x"],
        df_cargas_plot["y"],
        s=42,
        marker="o",
        facecolor="#FF7B00",
        edgecolor="#801500",
        linewidth=0.7,
        alpha=1.0,
        label=f"Residências BT ({len(df_cargas_plot)} domicílios)",
        zorder=4,
    )

if not df_pvs_plot.empty:
    ax_2s.scatter(
        df_pvs_plot["x"],
        df_pvs_plot["y"],
        s=50,
        marker="s",
        color="#00C853",
        edgecolor="#004D1A",
        linewidth=0.7,
        alpha=1.0,
        zorder=5,
        label=f"Geração Solar GD FV ({len(df_pvs_plot)} PVs)",
    )

if not df_trafos_plot.empty:
    ax_2s.scatter(
        df_trafos_plot["x"],
        df_trafos_plot["y"],
        s=115,
        marker="^",
        color="#7719FF",
        edgecolor="#3B0780",
        linewidth=0.9,
        alpha=1.0,
        zorder=6,
        label=f"Transformadores MT/BT ({len(df_trafos_plot)} trafos)",
    )

ax_2s.scatter([0], [0], s=450, marker="*", color="#990000", edgecolor="black", linewidth=1.4, zorder=8, label="Subestação SE (13,8 kV)")

ax_2s.autoscale(False)
ax_2s.imshow(img_sat, extent=extent_sat, origin="upper", alpha=ALPHA_SATELITE, zorder=0)
ax_2s.set_aspect("equal", adjustable="box")
ax_2s.set_xlim(xlim_sat)
ax_2s.set_ylim(ylim_sat)

ax_2s.set_title("Alocação Espacial de Equipamentos e Cargas", fontweight="bold", pad=12, fontsize=15.5)
ax_2s.set_xlabel("Distância Leste–Oeste relativa à Subestação (km)", fontweight="bold", fontsize=15)
ax_2s.set_ylabel("Distância Norte–Sul relativa à Subestação (km)", fontweight="bold", fontsize=15)
ax_2s.legend(loc="lower left", frameon=True, fontsize=12, framealpha=0.95, edgecolor="black", labelspacing=1.0, handletextpad=0.8, markerscale=0.85, borderpad=1.0)
estilizar_eixos_topologia(ax_2s, sob_satelite=True)

plt.tight_layout()
salvar_figura_topologia(fig_1b_sat, "fig1b_alocacao_equipamentos_alimentador_satelite")
plt.show()
print("Figuras 1a e 1b (com Satélite Esri World Imagery) salvas com sucesso em 600 DPI (PNG e PDF)!")

Salvo em Topologia: fig1a_topologia_barras_alimentador_satelite.png e fig1a_topologia_barras_alimentador_satelite.pdf (600 DPI)


Salvo em Topologia: fig1b_alocacao_equipamentos_alimentador_satelite.png e fig1b_alocacao_equipamentos_alimentador_satelite.pdf (600 DPI)


Figuras 1a e 1b (com Satélite Esri World Imagery) salvas com sucesso em 600 DPI (PNG e PDF)!


## Módulo 3: Alocação Espacial da Frota de Veículos Elétricos - Cenário 50% (Figura 2)

Dispersão espacial estocástica dos 474 veículos elétricos (2.581,2 kW instalados) no cenário de 50% de adoção residencial:
- **Semente de Monte Carlo:** **Seed 2026** (conforme alocação do arquivo `ESB01S4_evs.dss`);
- **Adoção Residencial Estrita:** Cada um dos 474 VEs pertence a um domicílio/residência único (1 VE por domicílio);
- **Barra com 1 VE (Círculos Verdes Claros):** Postes/barras onde apenas 1 domicílio conectado possui VE (3,6 kW ou 7,2 kW);
- **Barra com Múltiplos VEs (Círculos Maiores em Verde Escuro):** Postes/barras onde 2 ou mais domicílios vizinhos atendidos pelo mesmo ponto físico possuem VE (cada um com seu próprio VE, acumulando até 21,6 kW no poste);
- **Background Suave:** Linhas MT/BT em segundo plano translúcido para evidenciar as manchas e polos de demanda de recarga veicular;
- **Pontos Críticos:** Conexão espacial dos VEs com o transformador crítico `TRF_29L2218A` e a barra de maior afundamento de tensão `3864601538458035`.

In [16]:
# FIGURA 2: Alocação Espacial da Frota de Veículos Elétricos - Cenário 50% (Seed 2026)
caminho_ev_dss = PROJECT_ROOT / "data" / "rede" / "ESB01S4" / "ESB01S4_evs.dss"
evs_data = []
seed_identificada = "2026"
if caminho_ev_dss.exists():
    with open(caminho_ev_dss, "r", encoding="utf-8") as f:
        for l in f:
            if "semente (seed):" in l.lower() or "seed:" in l.lower():
                m_seed = re.search(r"seed[\):]?\s*([0-9]+)", l, re.I)
                if m_seed:
                    seed_identificada = m_seed.group(1)
            if "new load.ev_" in l.lower():
                m_bus = re.search(r"bus1=([^\s]+)", l, re.I)
                m_kw = re.search(r"kw=([^\s]+)", l, re.I)
                if m_bus:
                    b_ev = m_bus.group(1).split(".")[0].lower()
                    kw = float(m_kw.group(1)) if m_kw else 7.2
                    if b_ev in coords_km:
                        evs_data.append({"bus": b_ev, "kw": kw, "x": coords_km[b_ev][0], "y": coords_km[b_ev][1]})
df_evs_plot = pd.DataFrame(evs_data)

# Agrupamento por barra para destacar concentração de VEs
df_ev_bus = df_evs_plot.groupby("bus").agg(
    kw_total=("kw", "sum"),
    qtd_evs=("kw", "count"),
    x=("x", "first"),
    y=("y", "first"),
).reset_index()

df_ev_single = df_ev_bus[df_ev_bus["qtd_evs"] == 1]
df_ev_multi = df_ev_bus[df_ev_bus["qtd_evs"] > 1]

fig_b, ax_b = plt.subplots(figsize=(11.5, 14.2), dpi=600)

# Linhas de Baixa Tensão (BT) - Em cinza/azul claro suave (background)
for xs, ys in bt_segments:
    ax_b.plot(xs, ys, color="#B0C4DE", linewidth=0.75, alpha=0.45, zorder=1)

# Linhas de Média Tensão (MT) - Em vermelho translúcido (background)
for idx, (xs, ys) in enumerate(mt_segments):
    lbl = "Rede MT (Tronco 13,8 kV)" if idx == 0 else ""
    ax_b.plot(xs, ys, color="#E45756", linewidth=1.3, alpha=0.55, label=lbl, zorder=2)

# Transformadores MT/BT discretos
if not df_trafos_plot.empty:
    ax_b.scatter(
        df_trafos_plot["x"],
        df_trafos_plot["y"],
        s=48,
        marker="^",
        color="#BDBDBD",
        edgecolor="#424242",
        linewidth=0.7,
        alpha=0.75,
        zorder=3,
        label=f"Transformadores MT/BT ({len(df_trafos_plot)} un.)",
    )

# Subestação SE Eusébio
ax_b.scatter(
    [0],
    [0],
    s=280,
    marker="*",
    color="#990000",
    edgecolor="black",
    linewidth=1.2,
    zorder=4,
    label="Subestação SE Eusébio (13,8 kV)",
)

# Veículos Elétricos - Barras com 1 VE
ax_b.scatter(
    df_ev_single["x"],
    df_ev_single["y"],
    s=38,
    marker="o",
    color="#2CA02C",
    edgecolor="white",
    linewidth=0.5,
    alpha=0.85,
    zorder=5,
    label=f"Barra com 1 VE ({len(df_ev_single)} barras - 1 domicílio)",
)

# Veículos Elétricos - Barras com múltiplos VEs (maior concentração)
if not df_ev_multi.empty:
    ax_b.scatter(
        df_ev_multi["x"],
        df_ev_multi["y"],
        s=90,
        marker="o",
        color="#006837",
        edgecolor="#2CA02C",
        linewidth=1.8,
        alpha=0.95,
        zorder=6,
        label=f"Barra com Múltiplos VEs ({len(df_ev_multi)} barras com 2+ domicílios - até {df_ev_multi['kw_total'].max():.1f} kW)",
    )

# Destaque ao Transformador Crítico TRF_29L2218A
if trf_crit_coord:
    ax_b.scatter([trf_crit_coord[0]], [trf_crit_coord[1]], s=170, facecolors="none", edgecolors="#B2182B", linewidth=2.0, zorder=7)
    ax_b.annotate(
        f"Trafo Crítico: {trafo_mais_critico}\n(75 kVA | Sobrecarga 244% provocada por VEs)",
        xy=trf_crit_coord,
        xytext=(trf_crit_coord[0] - 1.15, trf_crit_coord[1] + 0.35),
        arrowprops=dict(facecolor="#B2182B", shrink=0.08, width=1.2, headwidth=6),
        fontsize=10.5,
        fontweight="bold",
        color="#B2182B",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#FFF5F0", edgecolor="#B2182B", alpha=0.92),
        zorder=9,
    )

# Destaque à Barra Crítica BT
if b_crit_coord:
    ax_b.annotate(
        f"Barra Crítica BT: {info_barra['bus']}\n({info_barra['distancia_km']:.2f} km da SE | V_min=0,786 pu)",
        xy=b_crit_coord,
        xytext=(b_crit_coord[0] + 0.35, b_crit_coord[1] - 0.55),
        arrowprops=dict(facecolor="#D7191C", shrink=0.08, width=1.2, headwidth=6),
        fontsize=10.5,
        fontweight="bold",
        color="#990000",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="#FFFFD4", edgecolor="#D7191C", alpha=0.92),
        zorder=9,
    )

# Caixa de Metadados e Parâmetros da Simulação de VEs
total_kw_evs = df_evs_plot["kw"].sum()
qtd_total_evs = len(df_evs_plot)
texto_info_ev = (
    f"Parâmetros da Frota de VEs:\n"
    f"• Penetração Domiciliar : 50,0%\n"
    f"• Veículos Alocados     : {qtd_total_evs} VEs\n"
    f"• Potência Instalada    : {total_kw_evs:,.1f} kW\n"
    f"• Semente Monte Carlo   : Seed {seed_identificada}\n"
    f"• Domicílios Atendidos  : 947 residências"
)
ax_b.text(
    0.98,
    0.03,
    texto_info_ev,
    transform=ax_b.transAxes,
    fontsize=11.0,
    fontfamily="monospace",
    verticalalignment="bottom",
    horizontalalignment="right",
    bbox=dict(boxstyle="round,pad=0.5", facecolor="#F7FCF5", edgecolor="#2CA02C", alpha=0.92),
    zorder=9,
)

ax_b.set_title(
    "Alocação Espacial Residencial da Frota de Veículos Elétricos - Alimentador ESB01S4 (Eusébio/CE)\n" 
    f"Dispersão Estocástica de {qtd_total_evs} VEs ({total_kw_evs:,.0f} kW) no Cenário de 50% de Adoção BT | Semente: Seed {seed_identificada}",
    fontweight="bold",
    pad=14,
    fontsize=14.0,
)
ax_b.set_xlabel("Distância Leste–Oeste relativa à Subestação (km)", fontweight="bold", fontsize=15)
ax_b.set_ylabel("Distância Norte–Sul relativa à Subestação (km)", fontweight="bold", fontsize=15)
ax_b.legend(loc="upper left", frameon=True, fontsize=11.0, framealpha=0.95, edgecolor="black")
ax_b.set_aspect("equal", adjustable="datalim")
estilizar_eixos_topologia(ax_b, sob_satelite=False)

plt.tight_layout()
salvar_figura_topologia(fig_b, "fig2_alocacao_veiculos_eletricos_50pct")
plt.show()
print(f"Figura 2 (Alocação de VEs - 50% Seed {seed_identificada}) salva com sucesso em 600 DPI (PNG e PDF)!")

Salvo em Topologia: fig2_alocacao_veiculos_eletricos_50pct.png e fig2_alocacao_veiculos_eletricos_50pct.pdf (600 DPI)


Figura 2 (Alocação de VEs - 50% Seed 2026) salva com sucesso em 600 DPI (PNG e PDF)!
